# Linear Regression and the Open Asteroid Dataset
This dataset of over 800,000 known asteroids is collected from the Small Body Database maintained by the NASA's Jet Propulsion Laboratory (JPL). It contains information about each asteroid's orbit and known measurable properties. New entires are added daily and can be obtained through the NASA Open Data Portal: 
https://data.nasa.gov/dataset/jpl-small-body-database-browser 

or the JPL portal: 
https://ssd.jpl.nasa.gov/tools/sbdb_query.html

The dataset provided was obtained from Basu (2019) IJAECS, 6, 4, 2394-2835. (Feel free to use an up-to-date dataset instead!)

***The goal of this notebook is to train a linear regression model to predict the properties of asteroids given some subset of available observations.*** In particular, we will focus on estimating asteroid diameters, but other features may also be explored if desired.

In [ ]:
# Importing the packages required for this exercise
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

## Data Inspection

Before you can begin building a ML model, you must first understand the data you are given and how they relate to the overarching objective of your model. Visualizing the data is always a good first step!

### Objective: 
*Start by reading in the data as a DataFrame and printing off some information, to get a feel for the type of data contained in this set. Make special note of the datatypes each entry is given as. Feel free to plot the data to help visualize them.*

In [ ]:
### INPUT NEEDED ###
# Reading in the data
asteroids = pd.read_csv(#input your file path)

# Display the whole dataframe...
display(asteroids)

# ... or you can print a random sample to get a better feel for how much data is or is not available
asteroids.sample(10)

### Header descriptions

You should notice that the dataset contains 27 unique columns. Below is a brief description of what each header means. 
* **name**: *Object name/designation*
* **a**: *semi-major axis [AU]*
* **e**: *eccentricity*
* **G**: *magnitude slope parameter (helps predict the magnitude of brightness as a function of the solar phase angle)*
* **i**: *inclination with respect to the xy ecliptic plane*
* **om**: *longitude of the ascending node*
* **w**: *argument of perihelion*
* **q**: *perihelion distance [AU]*
* **ad**: *aphelion distance [AU]*
* **per_y**: *Orbital period (around the sun) [yrs]*
* **data-arc**: *data arc-span [d]*
* **condition_code**: *Orbit condition code (uncertainty in orbital parameters rated from 0-9, where 0 is well-known and 9 is poorly-constrained)*
* **n_obs_used**: *number of observations used*
* **H**: *absolute magnitude parameter (brightness it would have if observed 1 AU from Sun)*
* **diameter**: *asteroid diameter [km]*
* **extent**: *object bi/tri-axial ellipsoid dimensions [km]*
* **albedo**: *geometric albedo (reflectiveness on scale of 0-1, where 1 = perfect light reflection)*
* **rot_per**: *rotation period (around rotational axis) [hrs]*
* **GM**: *standard gravitational parameter (mass times gravitational constant)*
* **BV**: *color index Bmag-Vmag*
* **UB**: *color index Umag-Bmag*
* **IR**: *color index Imag-Rmag*
* **spec_B**: *spectral taxonomic type (SMASSII)*
* **spec_T**: *spectral taxonomic type (Tholen)*
* **neo**: *Near Earth Object flag (bool)*
* **pha**: *Physically Hazardous Asteroid flag (bool)*
* **moid**: *Earth Minimum Orbit Intersection Distance [AU]*

#### Orbital Parameters

The orbits of each asteroid around the sun are described in several ways within this dataset. 

The shape of an asteroid's orbit is described by the semi-major axis (a) and eccentricity (e). The eccentricity describes how circular (e = 0) or elliptical (0 < e < 1) the orbit is. The semi-major axis is its farthest distance from the center of the circle/ellipse. The semi-minor axis (closest distance from center) can be calculated as $b = a\sqrt{1-e^{2}}$, such that larger eccentricities lead to shorter semi-minor axes (could this be useful for our models?).

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/7/76/An_image_describing_the_semi-major_and_semi-minor_axis_of_ellipse.svg/960px-An_image_describing_the_semi-major_and_semi-minor_axis_of_ellipse.svg.png?utm_source=commons.wikimedia.org&utm_campaign=index&utm_content=thumbnail&_=20141026153810" width="500"/>

(Credit: Wikimedia Commons, Sae1962)

The distance between each asteroid and the sun is described through the perihelion and aphelion distances (the minimum and maximum distance along the semi-major axis of the object's orbit), as shown below: 

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/c/c5/Perihelion_aphelion_semimajor_axis.svg/960px-Perihelion_aphelion_semimajor_axis.svg.png?utm_source=commons.wikimedia.org&utm_campaign=index&utm_content=thumbnail&_=20210920195140" width="500"/>

(Credit: Wikimedia Commons, Maxmath12) 

The inclination (i), longitude of the ascending node (om), and argument of perihelion (w) together give the three-dimensional tilt in the orbit of each object, as shown below:

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/e/eb/Orbit1.svg/960px-Orbit1.svg.png?utm_source=commons.wikimedia.org&utm_campaign=index&utm_content=thumbnail&_=20230508033814" width="400"/>

(Credit: Wikipedia, Lasunncty) 

#### Brightness, Color, and Classifications

Asteroids and other small bodies may be classified in several ways based on parameters such as their brightness, reflectiveness, and colors. The SMASS and Tholen classifications are two such systems represented in this data set (spec_B and spec_T). The definition of each classification flag can be found here: https://en.wikipedia.org/wiki/Asteroid_spectral_types

The colors of objects are calculated by subtracting their absolute magnitudes measured within any 2 color filters. Represented in this dataset are  **BV** (blue vs. green), **UB** (ultra-violet vs. blue), and **IR** (infrared vs. red). Colors can be a useful means of measuring how objects interact (or emit) light, which may help determine what type of object it is. 

This example below shows a color-color diagram of small-body populations, including Kuiper belt objects (red), comets (cyan), and asteroids of different Tholen classifications (black and white circles, with classifications marked):

<img src="small-body_CCD.png" width="400"/>

(Credit: Filacchione et al. 2022)

The amount of light reflected by an asteroid is described by a combination of the absolute magnitude parameter (H) and the geometric albedo (albedo). The albedo of an object is the ratio of the actual brightness we observe and that of a perfectly reflecting object of the same size, largely dependent on the surface material of the asteroid. For example, Saturn and its moon Titan receive the same amount of sunlight, but because Titan has a lower albedo, it appears dimmer than Saturn: 

<img src="https://upload.wikimedia.org/wikipedia/commons/3/36/Titan_and_Saturn_-_May_6_2012_-_combined_%2835516187116%29.jpg" width="400"/>
(Credit: Kevin Gill) 


On the other hand, the absolute magnitude parameter (H) is the hypothetical brightness we would observe if the object was 1 AU from the Sun. The absolute magnitude parameter and albedo are intrinsically linked; the more reflective a given surface is, the brighter you would expect it to appear if for a given amount of sunlight.

## Feature Selection
In this notebook, we are building a model that can predict the diameter of a given asteroid using linear regression. Based on the information provided above and within the dataset, which properties (features) would you expect to be most strongly linked to asteroid diameter? Those are the features you should start with for your model. 

### Objective: 
*Decide which features to use in your model and add them to a list.*

*NOTE: Obviously, "diameter" cannot be among these headers, since we will be using it as our label! Do not use the "extent" header either, as this is a direct description of the object's shape and size. In the spirit of this exercise, only use properties that are not directly a measurement of size.*

In [ ]:
### INPUT NEEDED ###
# Set up masks (lists) to help select the features of interest
feat_list = 
label = 

## Cleaning the Data

You will notice that not every asteroid in the dataset contains valid numbers for every feature, which can make the model difficult to create. In order to set up your training and test sets, you must first figure out how to handle of all of the "bad" data -- those that contain NaNs or no values. 

### Objective:
*Based on the features selected, determine what subset of the full dataset is appropriate. Remove any entry that contains NaNs (or handle them in some other way that does not disturb the ML process), and create a DataFrame containing only the data you will use in training/testing your model. Check your resulting dataset(s) to make sure it looks as expected.*

In [ ]:
# Your code here


## First-pass model

Now that you have a clean dataset containing only the information you will need to develop your model, it's time to build and train your linear regression model! To begin, we will first attempt to train a model using only the data provided within the (cleaned) dataset, without making any alterations to the data (later, we'll see how strategic data manipulation can help improve models). Then, we will test the newly-trained model to see how well it performs. 

### Objectives: 
- *Create and train a linear regression model based only on the features selected. Perform some statistical analyses to determine how well your model is performing based on the training set data.*

- *Test your model on the test set. How accurate are your estimated diameter values, compared to the actual asteroid diameters? Where does your model do well, and where does it fail?*

In [ ]:
### INPUT NEEDED ###
# Building the training and test set
feat_train, feat_test, label_train, label_test = train_test_split( , # feature DataFrame or Array, 
                                                                   , # labels DataFrame or Array
                                                                  test_size = , # fractional size of your test set (or use train_size)
                                                                  random_state= ) # controls random number generator for consistent compilations

### Check your resulting DataFrames to make sure it worked as intended ###
#
#

### INPUT NEEDED ###
# Creating the linear regression model
model = 

# Training the model
model.fit( #insert the training data

# Print results!
print(f"\nMODEL RESULTS\n-------------\nIntercept: {model.intercept_}\nSlopes: {[f"{round(model.coef_[0][i],3)} ({feat_list[i]})" for i in range(len(feat_list))]}")
        
### INPUT NEEDED ###
# Test your model with your test set
predict_test = model.predict(# insert your training set


### EVALUATE YOUR MODEL ###
# Print off your errors, RMSE, and r2 values
# Compare your expected and predicted values
#
#

## Improving model with feature engineering

You may notice that the model you've created does well for some asteroids, but is hugely inaccurate for others. This may suggest that the features you've chosen may not be the optimal properties for modeling diameter, or that the features we have are not sufficient as they're given. To improve the model, we will experiment with developing new features that aren't explicitly in the given dataset. The process of using currently-available data to build new features to feed into a ML model is called **feature engineering** and is a powerful tool for developing stronger models. 


Before we start building new features, it is useful to determine which of your current selected features are statistically more important for predicting the correct asteroid diameter. A useful tool (one of many) is [`permutation_importance`](https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html) from `sklearn.inspection`. This function determines the average "importance" of each feature by applying random permutations to the features and evaluating how the model prediction changes in response. By printing the highest mean importance from `permutation_importance`, you can determine the feature that most strongly predicts the diameter; higher mean importance values indicate stronger correlations between the feature and the label. This can help us determine whether we need to change our feature list, and/or which features we should adjust using feature engineering. 

### Objective: 
*Determine which features are most important for determining your asteroid diameters (you may learn more about the `permutation_importance` function [here](https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html), or use a different strategy to assess importance). Are there features you should remove from or add to your list? Explore how using different features change the results of your models.*

In [ ]:
### INPUT NEEDED ###
print("\nFEATURE IMPORTANCE (mean +/- std)\n------------------\n")
results = permutation_importance(estimator= , # your trained model
                                 X = , # data on which the importrance will be computed
                                 y = , # targets for supervised learning
                                 #scoring = , # name of scoring mechanism to use
                                 n_repeats = , # number of permutations to apply
                                 #n_jobs = , # number of parallel jobs
                                 #sample_weight = # sample weights used in scoring
                                 #max_samples = # number of samples to draw from X in each repeat
                                 random_state=) # pseudo-random number generator

# Printing an ordered list of features by their importance to the model
# NOTE: depending on what (if any) scoring mechanism chosen above, this code may not work as intended
for i in results.importances_mean.argsort()[::-1]:
        print(f"{feat_list[i]:<8}"
              f"{results.importances_mean[i]:.3f}"
              f" +/- {results.importances_std[i]:.3f}")



### Objective: 
*Based on the information you've collected (or through experimentation), construct some new features and determine whether that improves the model's effectiveness. You may consider adding/subtracting/multiplying features together or performing other basic calculations to alter existing features. Use your new features list to train a new model and compare the results to your previous model.*

In [1]:
# Your code here

## (OPTIONAL) Testing on larger dataset
A more recent asteroid dataset has been compiled here: https://www.kaggle.com/datasets/sakhawat18/asteroid-dataset/data

Does using this dataset, which may contain valid data for a larger sample of asteroids, improve your model? 

*NOTE: this dataset does not include all of the same headers as the original, so certain ML models may not perform well, depending on how you chose to build your model. Does using only the features in this larger dataset result in a more accurate model?*


In [ ]:
# Your code here

## Polynomial Regression 

You may notice that performing exponential operations on certain features result in better models than keeping all features in their linear state. This may hint towards the fact that an accurate model may not be linear at all. Can you model be improved by deviating from linear regression models? 

We can explore polynomial regression using `sklearn.preprocessing.PolynomialFeatures()` (for example, [as shown by this tutorial](https://www.kaggle.com/code/mahamedmahmoud/using-regression-diameter-classification-pha)). 

### Objective: 
*Transform your model into a polynomial and determine whether it returns a better fit than linear regression. What polynomial degree returns the best fit? How does it compare to your previous model(s)?*

*SUGGESTION: You may wish to experiment with tools such as `sklearn.model_selection.GridSearchCV` to optimize your polynomial.*

In [ ]:
from sklearn.preprocessing import PolynomialFeatures 

### INPUT NEEDED ###
# Create polynomial features
poly = PolynomialFeatures(# Define the polynomial you wish to apply to your data
# Apply the polynomial transformation to the data
feat_train_poly = poly.fit_transform(# features of the training set
feat_test_poly = poly.fit_transform(# features of the test set

# Initialize and train the model
poly_model = LinearRegression()
### INPUT NEEDED ###
poly_model.fit(# insert training data and labels

### TEST AND EVALUATE YOUR POLYNOMIAL MODEL ###
#
#

## Applying the ML model to virtual asteroids from a published model
We made some good progress building linear and polynomial regression models using the dataset provided, but you may have noticed that your cleaned dataset is much, much shorter than the full asteroid dataset. Unfortunately, a lack of good data may limit the effectiveness of any model you hope to develop and your ability to test it. One way around a lack of observational data for testing models is to generate a simulated dataset using theoretical models (though, note, this may introduce unintended biases into the model). Here we will generate a simulated population of asteroids. We will use this new set to determine whether your ML model can accurately recover the diameters of the input asteroids.

Our virtual asteroids will be generated based on the results of a recent paper, which approximated a relationship between diameter (in km), albedo (p), and absolute magnitude parameter (H): 

$D = \frac{1329}{\sqrt{p}}10^{-0.2H}$ 

[(Petrov 2025, Res. Notes AAS 9, 233)](https://iopscience.iop.org/article/10.3847/2515-5172/ae01a5/ampdf)


Before we use this to generate simulated asteroids, it's a good idea to double-check this researcher's work and inspect how well the model above represents the asteroids in the JPL dataset.

### Objective:
*Create a function that returns the diameter of an asteroid based on the equation above. Using that function, make a few plots that demonstrate how close/far the model is from the data results. Where does this published model perform best, and where does it struggle?*

*Are you satisfied with the results of the paper cited above? If not, is there anything you can do to improve it?*

In [ ]:
# Write a function based on the equation above that will return a diameter given an albedo and H value. 

def diameter(p, H): 
    """
    Returns a diameter or array of diameters of an asteroid given a measured 
    albedo and Absolute magnitude parameter. 
    
    PARAMETERS
    -----------
    p (float or array): Asteroid albedo 
    H (float or array): Absolute magnitude parameter
    
    RETURNS
    -----------
    D (float or array): Estimated asteroid diameter (units km)
    """
    
    ### INPUT NEEDED ###
    #
    #
    
    return D

In [ ]:
### Make plots to demonstrate how well this theoretical model describes actual asteroids ###
#
#

## Simulate an asteroid survey
Now that you have a working function for calculating asteroid diameters, you can use it randomly generate a large, simulated asteroid dataset. To do so, you will need to generate a set of albedos and absolute magnitude parameters, and then feed those values into your diameter function. You may also need to randomly generate values for other features you may have used in your own models, but it's worth noting that these values do not appear in the diameter calculations used by the theoretical model.

### Objective: 
*Simulate a dataset. For an added challenge, you may also add some simulated noise to the albedo and absolute magnitude parameter measurements to make the dataset more realistic. If your ML model uses features besides `p` and `H`, find a way to randomly generate reasonable values for those features for each simulated asteroid and add them to your virtual dataset.*

In [ ]:
# Your code here

## Test your model on the simulated survey

### Objective: 
*Run the simulated asteroid dataset through your ML model to determine how well it predicts asteroid diameter. You may use any (or all) versions of your model that you've developed throughout this workshop. Which version works best? What conclusions can you make about your model compared to the theoretical model?*

*NOTE: The order in which your features were read into the model while training is the order in which they must be fed into the model for predictions. Otherwise, you'll encounter an error.*

In [ ]:
# Your code here

# (OPTIONAL) Training a model on simulated data

While I personally wouldn't recommend training a ML model based on a theoretical model for real research purposes, attempting to do so as an experiment can be a useful exercise. Since you know what your model SHOULD look like, inspecting the results of your ML model can help you understand some of the processes happening within the ML black box. 

### Objective: 
*Train a new ML model on your simulated dataset. How well does the new model predict diameters now? How well does this new ML model predict the diameters of the real asteroid dataset? What changes do you need to make to your model-building process that gets you closer to the theoretical equation used to build your simulated set?*

In [ ]:
# Your code here